# HyperSense — Dataset Harmonization

## Objective

Transform the four extracted DHS datasets into a single harmonized hypertension dataset suitable for exploratory analysis, machine learning, and deployment.

### Input Datasets

- Ghana DHS 2014 Men (`ghana_men_extract.csv`)
- Ghana DHS 2014 Women (`ghana_women_extract.csv`)
- Benin DHS 2017–18 Men (`benin_men_extract.csv`)
- Benin DHS 2017–18 Women (`benin_women_extract.csv`)

### Harmonization Goals

Create a unified dataset with consistent:

- Variable names
- Coding schemes
- Data types
- Outcome definitions; across all four datasets.

### Key Tasks

1. Load extracted datasets.
2. Compare structures and variable availability.
3. Standardize variable names.
4. Harmonize tobacco-use variables.
5. Standardize BMI, blood pressure, and hypertension variables.
6. Ensure consistent coding for gender, education, and residence.
7. Concatenate all datasets into a single master dataframe.
8. Perform post-harmonization quality checks.
9. Export the final harmonized dataset for EDA and modeling.

### Important Notes

- Benin male respondents do not have anthropometric measurements; BMI is retained as missing.
- Benin blood pressure variables contained DHS special codes (994–999) which were recoded to missing during extraction.
- Hypertension status was derived using:
  - SBP ≥ 140 mmHg OR
  - DBP ≥ 90 mmHg
- Harmonization decisions are documented throughout this notebook to ensure reproducibility.

### Expected Output

`hypersense_harmonized.csv`

A single harmonized adult hypertension dataset combining Ghana and Benin DHS respondents.

In [144]:
import pandas as pd
import pyreadstat

In [145]:
gm = pd.read_csv("../outputs/ghana_men_extract.csv")
gw = pd.read_csv("../outputs/ghana_women_extract.csv")
bm = pd.read_csv("../outputs/benin_men_extract.csv")
bw = pd.read_csv("../outputs/benin_women_extract.csv")

In [146]:
print(gm.shape)
print(gw.shape)
print(bm.shape)
print(bw.shape)

(4388, 23)
(9396, 23)
(7595, 15)
(15928, 21)


In [147]:
# =====================================================
# Step 1 — Standardize Variable Names
# =====================================================

In [148]:
# Quick comparison of columns

print("GM:", gm.columns.tolist())
print("GW:", gw.columns.tolist())
print("BM:", bm.columns.tolist())
print("BW:", bw.columns.tolist())

GM: ['MCASEID', 'MV001', 'MV002', 'MV003', 'MV005', 'MV012', 'MV106', 'MV025', 'MV463A', 'MV463B', 'MV463C', 'MV463D', 'MV463X', 'MV463Z', 'BMI', 'SM500C1', 'SM500C2', 'SM862A', 'SM862B', 'gender', 'mean_sbp', 'mean_dbp', 'htn_status']
GW: ['CASEID', 'V001', 'V002', 'V003', 'V005', 'V012', 'V106', 'V025', 'V463A', 'V463B', 'V463C', 'V463D', 'V463X', 'V463Z', 'V445', 'S600CA', 'S600CB', 'S1056A', 'S1056B', 'gender', 'mean_sbp', 'mean_dbp', 'htn_status']
BM: ['MCASEID', 'MV001', 'MV002', 'MV003', 'MV005', 'MV012', 'MV106', 'MV025', 'MV463AA', 'MV463AB', 'SM936S', 'SM936D', 'BMI', 'gender', 'htn_status']
BW: ['CASEID', 'V001', 'V002', 'V003', 'V005', 'V012', 'V106', 'V025', 'V463A', 'V463B', 'V463C', 'V463D', 'V463X', 'V463Z', 'V463AA', 'V463AB', 'V445', 'S1435S', 'S1435D', 'gender', 'htn_status']


In [149]:
# Standardize Ghana Men variable names

gm = gm.rename(columns={
    "MCASEID": "caseid",
    "MV001": "cluster",
    "MV002": "household",
    "MV003": "respondent_number",
    "MV005": "sample_weight",
    "MV012": "age",
    "MV106": "educational_level",
    "MV025": "residence",
    "MV463A": "smokes_cig",
    "MV463B": "smokes_pipe",
    "MV463C": "chews_tobacco",
    "MV463D": "uses_snuff",
    "MV463X": "smokes_other",
    "MV463Z": "smokes_none",
    "BMI": "bmi",
    "SM500C1": "sbp_1",
    "SM500C2": "dbp_1",
    "SM862A": "sbp_2",
    "SM862B": "dbp_2",
    "gender": "gender",
    "mean_sbp": "mean_sbp",
    "mean_dbp": "mean_dbp",
    "htn_status": "htn_status",
})

In [150]:
print(len(gm.columns.tolist()))
print(gm.columns.tolist())
gm.head()

23
['caseid', 'cluster', 'household', 'respondent_number', 'sample_weight', 'age', 'educational_level', 'residence', 'smokes_cig', 'smokes_pipe', 'chews_tobacco', 'uses_snuff', 'smokes_other', 'smokes_none', 'bmi', 'sbp_1', 'dbp_1', 'sbp_2', 'dbp_2', 'gender', 'mean_sbp', 'mean_dbp', 'htn_status']


,caseid,cluster,household,respondent_number,sample_weight,age,educational_level,residence,smokes_cig,smokes_pipe,...,smokes_none,bmi,sbp_1,dbp_1,sbp_2,dbp_2,gender,mean_sbp,mean_dbp,htn_status
0,1 1 1,1.0,1.0,1.0,856663.0,50.0,2.0,2.0,0.0,0.0,...,1.0,1703.0,152.0,106.0,154.0,101.0,Male,153.0,103.5,1.0
1,1 3 1,1.0,3.0,1.0,856663.0,27.0,2.0,2.0,0.0,0.0,...,1.0,2048.0,133.0,79.0,136.0,82.0,Male,134.5,80.5,0.0
2,1 6 1,1.0,6.0,1.0,856663.0,24.0,2.0,2.0,0.0,0.0,...,1.0,2108.0,109.0,63.0,108.0,63.0,Male,108.5,63.0,0.0
3,111 1,1.0,11.0,1.0,856663.0,40.0,1.0,2.0,0.0,0.0,...,0.0,1972.0,117.0,84.0,117.0,82.0,Male,117.0,83.0,0.0
4,119 1,1.0,19.0,1.0,856663.0,43.0,2.0,2.0,0.0,0.0,...,1.0,2300.0,131.0,85.0,121.0,80.0,Male,126.0,82.5,0.0


In [151]:
# Standardize Ghana Women variable names

gw = gw.rename(columns={
    "CASEID": "caseid",
    "V001": "cluster",
    "V002": "household",
    "V003": "respondent_number",
    "V005": "sample_weight",
    "V012": "age",
    "V106": "educational_level",
    "V025": "residence",
    "V463A": "smokes_cig",
    "V463B": "smokes_pipe",
    "V463C": "chews_tobacco",
    "V463D": "uses_snuff",
    "V463X": "smokes_other",
    "V463Z": "smokes_none",
    "V445": "bmi",
    "S600CA": "sbp_1",
    "S600CB": "dbp_1",
    "S1056A": "sbp_2",
    "S1056B": "dbp_2",
    "gender": "gender",
    "mean_sbp": "mean_sbp",
    "mean_dbp": "mean_dbp",
    "htn_status": "htn_status",
})

In [152]:
print(len(gw.columns.tolist()))
print(gw.columns.tolist())
gw.head()

23
['caseid', 'cluster', 'household', 'respondent_number', 'sample_weight', 'age', 'educational_level', 'residence', 'smokes_cig', 'smokes_pipe', 'chews_tobacco', 'uses_snuff', 'smokes_other', 'smokes_none', 'bmi', 'sbp_1', 'dbp_1', 'sbp_2', 'dbp_2', 'gender', 'mean_sbp', 'mean_dbp', 'htn_status']


,caseid,cluster,household,respondent_number,sample_weight,age,educational_level,residence,smokes_cig,smokes_pipe,...,smokes_none,bmi,sbp_1,dbp_1,sbp_2,dbp_2,gender,mean_sbp,mean_dbp,htn_status
0,1 1 2,1.0,1.0,2.0,858896.0,42.0,1.0,2.0,0.0,0.0,...,1.0,2026.0,115.0,66.0,118.0,63.0,Female,116.5,64.5,0.0
1,1 3 2,1.0,3.0,2.0,858896.0,25.0,2.0,2.0,0.0,0.0,...,1.0,2360.0,121.0,90.0,117.0,91.0,Female,119.0,90.5,1.0
2,1 5 2,1.0,5.0,2.0,858896.0,37.0,1.0,2.0,0.0,0.0,...,1.0,NaN,104.0,71.0,94.0,66.0,Female,99.0,68.5,0.0
3,1 5 3,1.0,5.0,3.0,858896.0,15.0,1.0,2.0,0.0,0.0,...,1.0,NaN,127.0,85.0,124.0,85.0,Female,125.5,85.0,0.0
4,1 6 2,1.0,6.0,2.0,858896.0,19.0,2.0,2.0,0.0,0.0,...,1.0,2620.0,88.0,69.0,107.0,67.0,Female,97.5,68.0,0.0


In [153]:
# Standardize Benin Men variable names

bm = bm.rename(columns={
    "MCASEID": "caseid",
    "MV001": "cluster",
    "MV002": "household",
    "MV003": "respondent_number",
    "MV005": "sample_weight",
    "MV012": "age",
    "MV106": "educational_level",
    "MV025": "residence",
    "MV463AA": "freq_smoke_tobacco",
    "MV463AB": "freq_smokeless_tobacco",
    "BMI": "bmi",
    "SM936S": "mean_sbp",
    "SM936D": "mean_dbp",
    "gender": "gender",
    "htn_status": "htn_status",
})

In [154]:
print(len(bm.columns.tolist()))
print(bm.columns.tolist())
bm.head()

15
['caseid', 'cluster', 'household', 'respondent_number', 'sample_weight', 'age', 'educational_level', 'residence', 'freq_smoke_tobacco', 'freq_smokeless_tobacco', 'mean_sbp', 'mean_dbp', 'bmi', 'gender', 'htn_status']


,caseid,cluster,household,respondent_number,sample_weight,age,educational_level,residence,freq_smoke_tobacco,freq_smokeless_tobacco,mean_sbp,mean_dbp,bmi,gender,htn_status
0,1 3 1,1.0,3.0,1.0,1130086.0,31.0,0.0,2.0,0.0,1.0,106.0,66.0,NaN,Male,0.0
1,1 3 8,1.0,3.0,8.0,1130086.0,15.0,0.0,2.0,0.0,1.0,NaN,NaN,NaN,Male,NaN
2,1 15 3,1.0,15.0,3.0,1130086.0,45.0,0.0,2.0,0.0,0.0,124.0,89.0,NaN,Male,0.0
3,1 15 9,1.0,15.0,9.0,1130086.0,42.0,0.0,2.0,0.0,0.0,115.0,80.0,NaN,Male,0.0
4,1 15 11,1.0,15.0,11.0,1130086.0,15.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,Male,NaN


In [155]:
# Standardize Benin Women variable names

bw = bw.rename(columns={
    "CASEID": "caseid",
    "V001": "cluster",
    "V002": "household",
    "V003": "respondent_number",
    "V005": "sample_weight",
    "V012": "age",
    "V106": "educational_level",
    "V025": "residence",
    "V463A": "smokes_cig",
    "V463B": "smokes_pipe",
    "V463C": "chews_tobacco",
    "V463D": "uses_snuff",
    "V463X": "smokes_other",
    "V463Z": "smokes_none",
    "V463AA": "freq_smoke_tobacco",
    "V463AB": "freq_smokeless_tobacco",
    "V445": "bmi",
    "S1435S": "mean_sbp",
    "S1435D": "mean_dbp",
    "gender": "gender",
    "htn_status": "htn_status",
})

In [156]:
print(len(bw.columns.tolist()))
print(bw.columns.tolist())
bw.head()

21
['caseid', 'cluster', 'household', 'respondent_number', 'sample_weight', 'age', 'educational_level', 'residence', 'smokes_cig', 'smokes_pipe', 'chews_tobacco', 'uses_snuff', 'smokes_other', 'smokes_none', 'freq_smoke_tobacco', 'freq_smokeless_tobacco', 'bmi', 'mean_sbp', 'mean_dbp', 'gender', 'htn_status']


,caseid,cluster,household,respondent_number,sample_weight,age,educational_level,residence,smokes_cig,smokes_pipe,...,uses_snuff,smokes_other,smokes_none,freq_smoke_tobacco,freq_smokeless_tobacco,bmi,mean_sbp,mean_dbp,gender,htn_status
0,1 3 2,1.0,3.0,2.0,1134416.0,30.0,0.0,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,NaN,101.0,65.0,Female,0.0
1,1 3 4,1.0,3.0,4.0,1134416.0,23.0,0.0,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,Female,NaN
2,1 9 2,1.0,9.0,2.0,1134416.0,33.0,0.0,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,2480.0,NaN,NaN,Female,NaN
3,1 9 8,1.0,9.0,8.0,1134416.0,39.0,0.0,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,2395.0,NaN,NaN,Female,NaN
4,1 9 15,1.0,9.0,15.0,1134416.0,33.0,0.0,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,3289.0,NaN,NaN,Female,NaN


In [157]:
# =====================================================
# Add missing variables so all datasets share schema
# =====================================================

# ---------- Ghana Men ----------
gm["freq_smoke_tobacco"] = pd.NA
gm["freq_smokeless_tobacco"] = pd.NA


# ---------- Ghana Women ----------
gw["freq_smoke_tobacco"] = pd.NA
gw["freq_smokeless_tobacco"] = pd.NA


# ---------- Benin Men ----------

# Tobacco type variables not collected
bm["smokes_cig"] = pd.NA
bm["smokes_pipe"] = pd.NA
bm["chews_tobacco"] = pd.NA
bm["uses_snuff"] = pd.NA
bm["smokes_other"] = pd.NA
bm["smokes_none"] = pd.NA

# BMI not collected
bm["bmi"] = pd.NA

# Individual BP readings unavailable
bm["sbp_1"] = pd.NA
bm["sbp_2"] = pd.NA
bm["dbp_1"] = pd.NA
bm["dbp_2"] = pd.NA


# ---------- Benin Women ----------

# Individual BP readings unavailable
bw["sbp_1"] = pd.NA
bw["sbp_2"] = pd.NA
bw["dbp_1"] = pd.NA
bw["dbp_2"] = pd.NA

In [158]:
# =====================================================
# Country identifiers
# =====================================================

gm["country"] = "Ghana"
gw["country"] = "Ghana"

bm["country"] = "Benin"
bw["country"] = "Benin"

In [159]:
# =====================================================
# Final harmonized column order
# =====================================================

final_columns = [

    "caseid",
    "cluster",
    "household",
    "respondent_number",
    "sample_weight",

    "age",
    "educational_level",
    "residence",

    "smokes_cig",
    "smokes_pipe",
    "chews_tobacco",
    "uses_snuff",
    "smokes_other",
    "smokes_none",

    "freq_smoke_tobacco",
    "freq_smokeless_tobacco",

    "bmi",

    "sbp_1",
    "sbp_2",
    "dbp_1",
    "dbp_2",

    "mean_sbp",
    "mean_dbp",

    "gender",
    "country",

    "htn_status"
]

In [160]:
# =====================================================
# Reorder columns
# =====================================================

gm = gm[final_columns]
gw = gw[final_columns]
bm = bm[final_columns]
bw = bw[final_columns]

In [161]:
# =====================================================
# Verify schemas match
# =====================================================

print(gm.columns.equals(gw.columns))
print(gm.columns.equals(bm.columns))
print(gm.columns.equals(bw.columns))

print(gm.shape)
print(gw.shape)
print(bm.shape)
print(bw.shape)

True
True
True
(4388, 26)
(9396, 26)
(7595, 26)
(15928, 26)


In [162]:
# =====================================================
# Concatenate all datasets
# =====================================================

master_df = pd.concat(
    [gm, gw, bm, bw],
    ignore_index=True
)

print(master_df.shape)

(37307, 26)


C:\Users\USER\AppData\Local\Temp\ipykernel_17588\2444512119.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat(


In [163]:
# =====================================================
# Basic audit of harmonized dataset
# =====================================================

print(master_df.shape)

print("\nCountry distribution")
print(master_df["country"].value_counts())

print("\nGender distribution")
print(master_df["gender"].value_counts())

print("\nHTN distribution")
print(master_df["htn_status"].value_counts(dropna=False))

(37307, 26)

Country distribution
country
Benin    23523
Ghana    13784
Name: count, dtype: int64

Gender distribution
gender
Female    25324
Male      11983
Name: count, dtype: int64

HTN distribution
htn_status
0.0    18097
NaN    16861
1.0     2349
Name: count, dtype: int64


In [164]:
# =====================================================
# Missingness audit
# =====================================================

(
    master_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

sbp_2                     63.165090
dbp_2                     63.165090
dbp_1                     63.157048
sbp_1                     63.157048
bmi                       57.321682
mean_sbp                  45.195272
htn_status                45.195272
mean_dbp                  45.189911
freq_smoke_tobacco        36.947490
freq_smokeless_tobacco    36.947490
chews_tobacco             20.371512
smokes_pipe               20.371512
smokes_other              20.371512
uses_snuff                20.371512
smokes_cig                20.368832
smokes_none               20.368832
caseid                     0.000000
cluster                    0.000000
educational_level          0.000000
residence                  0.000000
sample_weight              0.000000
age                        0.000000
respondent_number          0.000000
household                  0.000000
gender                     0.000000
country                    0.000000
dtype: float64

In [165]:
# Missing HTN status by dataset
master_df.groupby(["country", "gender"])["htn_status"] \
         .apply(lambda x: x.isna().mean() * 100)

country  gender
Benin    Female    80.769714
         Male      52.113232
Ghana    Female     0.287356
         Male       0.250684
Name: htn_status, dtype: float64

In [166]:
# HTN prevalence among measured respondents
master_df.groupby(
    ["country", "gender"]
)["htn_status"].value_counts(normalize=True) * 100

country  gender  htn_status
Benin    Female  0.0           87.104146
                 1.0           12.895854
         Male    0.0           83.530382
                 1.0           16.469618
Ghana    Female  0.0           90.991568
                 1.0            9.008432
         Male    0.0           88.325337
                 1.0           11.674663
Name: proportion, dtype: float64

In [167]:
# Number of measured respondents by country and sex
master_df.groupby(
    ["country", "gender"]
)["htn_status"].count()

country  gender
Benin    Female    3063
         Male      3637
Ghana    Female    9369
         Male      4377
Name: htn_status, dtype: int64

In [168]:
# Raw counts of hypertensives
master_df.groupby(
    ["country", "gender"]
)["htn_status"].sum()

country  gender
Benin    Female    395.0
         Male      599.0
Ghana    Female    844.0
         Male      511.0
Name: htn_status, dtype: float64

In [169]:
# Check duplicates
master_df["caseid"].duplicated().sum()

np.int64(0)

In [170]:
# Final schema audit
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37307 entries, 0 to 37306
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   caseid                  37307 non-null  object 
 1   cluster                 37307 non-null  float64
 2   household               37307 non-null  float64
 3   respondent_number       37307 non-null  float64
 4   sample_weight           37307 non-null  float64
 5   age                     37307 non-null  float64
 6   educational_level       37307 non-null  float64
 7   residence               37307 non-null  float64
 8   smokes_cig              29708 non-null  float64
 9   smokes_pipe             29707 non-null  float64
 10  chews_tobacco           29707 non-null  float64
 11  uses_snuff              29707 non-null  float64
 12  smokes_other            29707 non-null  float64
 13  smokes_none             29708 non-null  float64
 14  freq_smoke_tobacco      23523 non-null

In [171]:
# Save harmonized master dataset
master_df.to_csv(
    "../outputs/hypersense_master.csv",
    index=False
)

print("HyperSense master dataset saved successfully.")
print(master_df.shape)

HyperSense master dataset saved successfully.
(37307, 26)


## Harmonization Complete

The four DHS datasets (Ghana Men, Ghana Women, Benin Men, and Benin Women) were successfully standardized and merged into a single harmonized analytical dataset. Variable names, coding structures, and target definitions were aligned across datasets, while dataset-specific variables and missing fields were handled appropriately to ensure schema consistency.

The final master dataset contains respondents from both countries and both sexes in a unified structure suitable for exploratory analysis and machine learning. Quality assurance checks were performed to verify schema alignment, sample counts, hypertension prevalence estimates, and expected missingness patterns arising from DHS blood pressure measurement subsampling.

This harmonized dataset serves as the foundation for all subsequent exploratory analysis, feature engineering, model development, and deployment activities within the HyperSense project.